# CounselChat Dataset Exploration

Loads **CounselChat** via `src.data_prep.load_counselchat()` (same normalization as the training pipeline). Dataset id: `nbertagnolli/counsel-chat`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_prep import load_counselchat

DATASET_ID = "nbertagnolli/counsel-chat"
df = load_counselchat(DATASET_ID)
print("shape:", df.shape)
print("columns:", list(df.columns))
print(df.head(3).to_string())


# Topic Distribution

Topic histogram and question/answer multiplicity (same dataframe as above).

In [ ]:
if "topic" not in df.columns:
    raise RuntimeError("Expected topic column from load_counselchat")

vc = df["topic"].value_counts()
print(vc.head(25).to_string())
print()

nq = df["question"].nunique()
per_q = df.groupby("question").size()
print("unique questions:", nq)
print("answers per question — min / median / max:", int(per_q.min()), float(per_q.median()), int(per_q.max()))


# Filtering Analysis

Uses `deduplicate_best_answer_per_question` and `filter_psychoeducation` from `src/data_prep.py`. Reference funnel from the paper: **2775 → 2612** (drop incomplete rows) → **863** (dedup) → **750** (psychoeducation filter).

In [ ]:
from src.data_prep import deduplicate_best_answer_per_question, filter_psychoeducation

n_load = len(df)
deduped = deduplicate_best_answer_per_question(df)
n_dedup = len(deduped)
filtered = filter_psychoeducation(deduped)
n_filt = len(filtered)

kf = set(zip(filtered["question"], filtered["answer"]))
removed = deduped[~deduped.apply(lambda r: (r["question"], r["answer"]) in kf, axis=1)]

print(f"Computed (this run): {n_load} → {n_dedup} → {n_filt}")
print("Paper funnel: 2775 → 2612 → 863 → 750")
print()
print("--- Examples REMOVED by psychoeducation filter (up to 2) ---")
if len(removed):
    sample_r = removed.head(2)
    for _, r in sample_r.iterrows():
        print("topic:", r.get("topic", ""))
        print("Q:", str(r["question"])[:200], "...")
        print("A:", str(r["answer"])[:200], "...")
        print()
else:
    print("(none)")
print("--- Examples KEPT after filter (up to 2) ---")
for _, r in filtered.head(2).iterrows():
    print("topic:", r.get("topic", ""))
    print("Q:", str(r["question"])[:200])
    print("A:", str(r["answer"])[:200])
    print()
